In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 62646)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 53155)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 44197)

# #### Device() ####
# device = cuda:6

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:6)
# edge_attr                (32464, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:6)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

---

In [8]:
from modules.layers import AttentionSetPooling
from modules.model import BaseAutoencoder
from modules.norm import logVST, zlogVST
from modules.train import Experiment, grid
from modules.trainers import ReconstrTrainer
from functools import partial
import torch.nn as nn

In [9]:
## Model

model_template = partial(
    BaseAutoencoder,
    dataset = dataset,
    method = 'node',

    # layers
    norm_class=zlogVST,
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,
    mlp=False,

    # layer params
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,
)

model_grid = grid(
    model_template,
    embed_dim=[16,32,64,128,256],
    hidden_dims=[0,1,2,3,4,5,6],
)

## Trainer

trainer_template = partial(
    ReconstrTrainer,
    norm_class=logVST
)

trainer_grid = grid(
    trainer_template,
    lr=[1e-1,1e-2,1e-3,1e-4,1e-5],
)

## Experiment

expt = Experiment(
    num_trials=1,
    num_epochs=50,
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

175


In [ ]:
expt.run_experiment(
    'reconstr_grid_1',
    report_metrics=['loss','rmse','mae','r2'],
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/175 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]